# Diffusion Model Training with LoRA — Healthy Brain MRI

## Overview
This notebook trains a **2D DDPM diffusion model** on healthy thin-slice T1w brain MRI
using **LoRA (Low-Rank Adaptation)** for parameter-efficient training.
The trained model is then used at inference time for **unsupervised anomaly detection**.

---

## Anomaly Detection Pipeline (inference)

```
Diseased slice  x₀
      │
      ▼
Saliency map from ACAT classifier
      → rough binary mask  M  of 'where pathology probably is'
      │
      ▼
DDIM encode  (forward noising, only inside masked region)
      x_L  =  q(x_L | x₀)   applied only where M = 1
      │
      ▼
Hybrid DDPM + DDIM decode
      ├─ inside  M = 1  →  DDPM steps  (stochastic, free to change pathology)
      └─ outside M = 0  →  DDIM steps  (deterministic, preserves healthy anatomy)
      │
      ▼
Healthy reconstruction  x̂₀
      │
      ▼
Anomaly map  =  | x₀ − x̂₀ |
```

**Key idea:** the diffusion model has never seen pathology during training.
When asked to reconstruct a diseased region (via DDPM decode inside the mask),
it generates the most likely *healthy* anatomy for that location.
The residual difference is the anomaly signal.

---

## LoRA rationale
The base `DiffusionModelUNet` has ~102 M parameters.
LoRA injects low-rank matrices `A ∈ R^{r×d}` and `B ∈ R^{d×r}` alongside
each attention projection (`Q, K, V, out`):

```
W' x  =  W x  +  (B A) x · (α / r)
```

Only `A` and `B` are trained (base weights frozen).
With rank `r = 4` targeting all attention linear layers, LoRA adds ~0.8 M
trainable parameters (<1 % of total), cutting GPU memory and training time
while retaining full model capacity at inference.

---

## Notebooks dependency
| Notebook | Role |
|----------|------|
| `04_thin_slice_preprocessing.ipynb` | `preprocess_volume`, `ThinSliceDataset`, splits |
| `02_model_setup.ipynb` | architecture reference (DiffusionModelUNet config) |

## 1 · Install dependencies

In [ ]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip_install('monai[all]>=1.3')
pip_install('einops', 'lpips')   # used internally by MONAI generative
print('Dependencies ready.')


## 2 · Imports

In [ ]:
import copy
import json
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from monai.networks.nets import DiffusionModelUNet
from monai.networks.schedulers import DDIMScheduler, DDPMScheduler
from monai.transforms import Compose, EnsureType, RandAffine, RandFlip, ScaleIntensity

# ── auto-locate brain_only directory ────────────────────────────────────
def _find_data_root(dirname='brain_only'):
    # 1. check cwd and every parent (works locally regardless of drive/path)
    for p in [Path.cwd(), *Path.cwd().parents]:
        candidate = p / dirname
        if candidate.is_dir():
            return candidate
    # 2. common RunPod / cloud paths
    for p in [Path('/workspace/brats'), Path('/workspace'), Path.home()]:
        candidate = p / dirname
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not find '{dirname}' directory. "
        f"Place the data folder next to this notebook or set DATA_ROOT manually."
    )

DATA_ROOT  = _find_data_root()
LOG_DIR    = Path.cwd() / 'logs'
CKPT_DIR   = LOG_DIR / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
(LOG_DIR / 'figures').mkdir(parents=True, exist_ok=True)

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DATA_ROOT: {DATA_ROOT}')
print(f'Device   : {DEVICE}')
print(f'PyTorch  : {torch.__version__}')

## 3 · Data pipeline

Re-defines `preprocess_volume`, `ThinSliceDataset`, and the train/val split
exactly as in `04_thin_slice_preprocessing.ipynb` so this notebook is self-contained.

In [ ]:
# ── subject discovery ───────────────────────────────────────────────────
def discover_thin_subjects(root, threshold_mm=2.0):
    subjects = []
    for child in sorted(root.iterdir()):
        nii = child / 't1_brain.nii.gz'
        if not (child.is_dir() and nii.exists()):
            continue
        img  = nib.load(str(nii))
        z_sp = float(np.sqrt((img.affine[:3, 2] ** 2).sum()))
        if z_sp < threshold_mm:
            subjects.append(dict(
                id=child.name, path=nii,
                z_sp=round(z_sp, 3),
                shape=tuple(img.header.get_data_shape()[:3]),
            ))
    return subjects


def split_subjects(subjects, train_frac=0.80, val_frac=0.10, seed=42):
    rng    = random.Random(seed)
    names  = sorted(s['id'] for s in subjects)
    rng.shuffle(names)
    lookup = {s['id']: s for s in subjects}
    ordered = [lookup[n] for n in names]
    n        = len(ordered)
    n_train  = int(n * train_frac)
    n_val    = int(n * val_frac)
    return ordered[:n_train], ordered[n_train:n_train+n_val], ordered[n_train+n_val:]


# ── slice preprocessing ──────────────────────────────────────────────────
def preprocess_volume(nii_path, target_size=256, z_low=0.15, z_high=0.85, fill_thresh=0.05):
    """
    Load a T1w NIfTI volume → list of valid (1, 256, 256) float32 tensors.
    Steps: percentile norm → Z-range filter → fill filter → resize.
    """
    vol = nib.load(str(nii_path)).get_fdata(dtype=np.float32)
    if vol.ndim != 3:
        return []
    nonzero = vol[vol != 0]
    if nonzero.size == 0:
        return []
    p_lo, p_hi = np.percentile(nonzero, [0.5, 99.5])
    vol = np.clip((vol - p_lo) / max(float(p_hi - p_lo), 1e-6), 0.0, 1.0)
    D   = vol.shape[2]
    z0, z1 = int(np.floor(D * z_low)), int(min(np.ceil(D * z_high), D))
    slices = []
    for zi in range(z0, z1):
        s = vol[:, :, zi]
        if np.count_nonzero(s) / s.size < fill_thresh:
            continue
        if s.shape[0] != target_size or s.shape[1] != target_size:
            t = torch.from_numpy(s[None, None])
            s = F.interpolate(t, (target_size, target_size),
                              mode='bilinear', align_corners=False).squeeze().numpy()
        slices.append(torch.from_numpy(s[np.newaxis].astype(np.float32)))
    return slices


# ── MONAI dataset ────────────────────────────────────────────────────────
class ThinSliceDataset(Dataset):
    def __init__(self, subjects, transform=None, target_size=256):
        self.subjects    = list(subjects)
        self.transform   = transform
        self.target_size = target_size

    def __len__(self):
        return len(self.subjects)

    def __getitem__(self, idx):
        entry  = self.subjects[idx]
        slices = preprocess_volume(entry['path'], target_size=self.target_size)
        if not slices:
            raise RuntimeError(f"No valid slices for {entry['id']}")
        tensor = random.choice(slices)
        if self.transform:
            tensor = self.transform(tensor)
        return tensor


# ── build splits & loaders ───────────────────────────────────────────────
all_thin                         = discover_thin_subjects(DATA_ROOT)
train_subjects, val_subjects, _  = split_subjects(all_thin)

train_transform = Compose([
    RandFlip(prob=0.5, spatial_axis=1),
    RandAffine(prob=0.5, rotate_range=(np.deg2rad(5),),
               translate_range=(5, 5), padding_mode='zeros', mode='bilinear'),
    ScaleIntensity(minv=0.0, maxv=1.0),
    EnsureType(dtype=torch.float32),
])
val_transform = Compose([EnsureType(dtype=torch.float32)])

BATCH_SIZE   = 8
train_loader = DataLoader(ThinSliceDataset(train_subjects, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, drop_last=True)
val_loader   = DataLoader(ThinSliceDataset(val_subjects, val_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train subjects : {len(train_subjects)}  |  batches/epoch : {len(train_loader)}')
print(f'Val   subjects : {len(val_subjects)}  |  batches/epoch : {len(val_loader)}')


## 4 · LoRA implementation

We inject LoRA into every `nn.Linear` layer inside the UNet attention blocks.
The `LoRALinear` wrapper keeps the frozen base weight and adds the trainable
`A` (down-projection) and `B` (up-projection) matrices:

```
output = W x  +  B(A(x)) · (alpha / r)
```

`inject_lora(model)` walks the module tree, replaces every `nn.Linear` inside
any module whose name contains `attn` or `attention`, and returns the count
of injected layers so we can verify coverage.

In [ ]:
class LoRALinear(nn.Module):
    """
    Drop-in replacement for nn.Linear with LoRA side-branch.

    During training only A and B are updated; W is frozen.
    At inference the effective weight is W + B@A * (alpha/r).
    """

    def __init__(self, linear: nn.Linear, r: int = 4, alpha: float = 4.0):
        super().__init__()
        self.r     = r
        self.scale = alpha / r
        in_f, out_f = linear.in_features, linear.out_features

        # frozen base weight & bias (no grad)
        self.weight = nn.Parameter(linear.weight.data.clone(), requires_grad=False)
        self.bias   = (nn.Parameter(linear.bias.data.clone(), requires_grad=False)
                       if linear.bias is not None else None)

        # trainable LoRA matrices
        self.lora_A = nn.Parameter(torch.empty(r, in_f))   # (r, in)
        self.lora_B = nn.Parameter(torch.zeros(out_f, r))  # (out, r)
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        # lora_B initialised to zero so initial output == base output

    def forward(self, x):
        base  = F.linear(x, self.weight, self.bias)
        lora  = F.linear(F.linear(x, self.lora_A), self.lora_B) * self.scale
        return base + lora

    def merge_weights(self):
        """Merge LoRA into base weight (for export / faster inference)."""
        with torch.no_grad():
            self.weight.data += (self.lora_B @ self.lora_A) * self.scale
        self.lora_A.requires_grad_(False)
        self.lora_B.requires_grad_(False)


def inject_lora(model: nn.Module, r: int = 4, alpha: float = 4.0,
                target_keywords=('attn', 'attention')) -> int:
    """
    Walk the model and replace every nn.Linear inside an attention block
    with LoRALinear.  Returns the number of layers replaced.
    """
    replaced = 0

    def _is_attn_module(name):
        return any(k in name.lower() for k in target_keywords)

    for mod_name, module in list(model.named_modules()):
        if not _is_attn_module(mod_name):
            continue
        for child_name, child in list(module.named_children()):
            if isinstance(child, nn.Linear):
                setattr(module, child_name,
                        LoRALinear(child, r=r, alpha=alpha).to(child.weight.device))
                replaced += 1

    return replaced


def lora_state_dict(model: nn.Module) -> dict:
    """Return only the trainable LoRA parameters (for lightweight checkpoints)."""
    return {k: v for k, v in model.state_dict().items()
            if 'lora_A' in k or 'lora_B' in k}


def count_parameters(model: nn.Module):
    total  = sum(p.numel() for p in model.parameters())
    train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - train
    return total, train, frozen


## 5 · Model, scheduler, LoRA injection

In [ ]:
# ── base UNet (same config as notebook 02) ──────────────────────────────
unet = DiffusionModelUNet(
    spatial_dims        = 2,
    in_channels         = 1,
    out_channels        = 1,
    num_channels        = (128, 256, 256, 512),
    attention_levels    = (False, False, True, True),
    num_res_blocks      = 2,
    num_head_channels   = 64,
).to(DEVICE)

# ── inject LoRA ──────────────────────────────────────────────────────────
LORA_RANK  = 4
LORA_ALPHA = 4.0
n_replaced = inject_lora(unet, r=LORA_RANK, alpha=LORA_ALPHA)
print(f'LoRA layers injected : {n_replaced}')

total, trainable, frozen = count_parameters(unet)
print(f'Total params     : {total/1e6:.2f} M')
print(f'Trainable (LoRA) : {trainable/1e6:.3f} M  ({100*trainable/total:.2f}%)')
print(f'Frozen (base)    : {frozen/1e6:.2f} M')

# ── schedulers ───────────────────────────────────────────────────────────
T = 1000   # total diffusion timesteps

ddpm_scheduler = DDPMScheduler(
    num_train_timesteps = T,
    beta_start          = 1e-4,
    beta_end            = 0.02,
    beta_schedule       = 'linear',
    clip_sample         = False,
)

ddim_scheduler = DDIMScheduler(
    num_train_timesteps = T,
    beta_start          = 1e-4,
    beta_end            = 0.02,
    beta_schedule       = 'linear',
    clip_sample         = False,
)
print('Schedulers ready.')


## 6 · Training configuration

In [ ]:
# ── optimiser: only LoRA params have requires_grad=True ─────────────────
LR          = 1e-4
N_EPOCHS    = 100       # increase for proper training (suggest 500+)
GRAD_CLIP   = 1.0
SAVE_EVERY  = 10        # checkpoint every N epochs
LOG_EVERY   = 1         # print loss every N epochs

lora_params = [p for p in unet.parameters() if p.requires_grad]
optimizer   = torch.optim.AdamW(lora_params, lr=LR, weight_decay=1e-4)
scheduler_lr = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=LR * 0.01
)

print(f'Optimising {len(lora_params)} LoRA parameter tensors')
print(f'Total trainable scalars : {sum(p.numel() for p in lora_params):,}')
print(f'Epochs : {N_EPOCHS}  |  LR : {LR}  |  Batch : {BATCH_SIZE}')


## 7 · Training loop

Standard DDPM training objective: predict the noise `ε` added at a random
timestep `t` and minimise `MSE(ε_pred, ε_true)`.

```
for each batch x₀:
    t   ~ Uniform(0, T-1)
    ε   ~ N(0, I)
    x_t = √ᾱ_t · x₀  +  √(1-ᾱ_t) · ε       (forward process)
    ε̂   = UNet(x_t, t)                        (noise prediction)
    loss = MSE(ε̂, ε)
```

In [ ]:
train_losses = []
val_losses   = []

unet.train()

for epoch in range(1, N_EPOCHS + 1):
    # ── TRAIN ─────────────────────────────────────────────────────────
    unet.train()
    epoch_loss = 0.0
    t0 = time.perf_counter()

    for batch in train_loader:
        x0 = batch.to(DEVICE)                          # (B, 1, 256, 256)

        # sample random timesteps
        t  = torch.randint(0, T, (x0.shape[0],), device=DEVICE).long()

        # sample noise
        noise = torch.randn_like(x0)

        # forward noising: x_t = sqrt(alpha_bar_t)*x0 + sqrt(1-alpha_bar_t)*noise
        x_t = ddpm_scheduler.add_noise(x0, noise, t)

        # predict noise
        noise_pred = unet(x_t, t)

        loss = F.mse_loss(noise_pred, noise)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(lora_params, GRAD_CLIP)
        optimizer.step()

        epoch_loss += loss.item()

    scheduler_lr.step()
    avg_train = epoch_loss / len(train_loader)
    train_losses.append(avg_train)

    # ── VAL ───────────────────────────────────────────────────────────
    unet.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            x0    = batch.to(DEVICE)
            t     = torch.randint(0, T, (x0.shape[0],), device=DEVICE).long()
            noise = torch.randn_like(x0)
            x_t   = ddpm_scheduler.add_noise(x0, noise, t)
            pred  = unet(x_t, t)
            val_loss += F.mse_loss(pred, noise).item()
    avg_val = val_loss / len(val_loader)
    val_losses.append(avg_val)

    # ── LOG ───────────────────────────────────────────────────────────
    elapsed = time.perf_counter() - t0
    if epoch % LOG_EVERY == 0:
        print(f'Epoch {epoch:04d}/{N_EPOCHS} | '
              f'train={avg_train:.5f}  val={avg_val:.5f} | '
              f'{elapsed:.1f}s | lr={scheduler_lr.get_last_lr()[0]:.2e}')

    # ── CHECKPOINT ────────────────────────────────────────────────────
    if epoch % SAVE_EVERY == 0:
        ckpt = {
            'epoch'       : epoch,
            'unet_full'   : unet.state_dict(),          # full weights
            'lora_only'   : lora_state_dict(unet),      # lightweight LoRA-only
            'optimizer'   : optimizer.state_dict(),
            'train_losses': train_losses,
            'val_losses'  : val_losses,
        }
        path = CKPT_DIR / f'diffusion_lora_epoch{epoch:04d}.pt'
        torch.save(ckpt, path)
        print(f'  Checkpoint saved → {path.name}')

print('Training complete.')


## 8 · Loss curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label='Train MSE', color='#5b9bd5')
ax.plot(val_losses,   label='Val MSE',   color='#e07b54')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE loss')
ax.set_title('Diffusion model (LoRA) training loss')
ax.legend()
plt.tight_layout()
fig.savefig(LOG_DIR / 'figures' / 'lora_loss_curve.png', dpi=150)
plt.show()

with open(LOG_DIR / 'lora_training_log.json', 'w') as f:
    json.dump({'train_losses': train_losses, 'val_losses': val_losses}, f, indent=2)
print('Loss log saved.')


## 9 · Inference — hybrid DDPM / DDIM decode

This implements the full anomaly detection pipeline described in the overview.

### `ddim_encode`
Runs the *deterministic* DDIM forward process from `t=0` to `t=L` in the masked
region only. Unmasked pixels keep their original values.

### `hybrid_decode`
Runs reverse diffusion from `t=L` to `t=0`:
- **Inside mask** (`M=1`): DDPM step — stochastic, allows the model to
  replace pathological content with healthy anatomy
- **Outside mask** (`M=0`): DDIM step — deterministic, preserves the
  original healthy-looking anatomy exactly

### `anomaly_map`
Pixel-wise absolute difference `|x₀ − x̂₀|` between original and reconstruction.

In [ ]:
@torch.no_grad()
def ddim_encode(x0: torch.Tensor, mask: torch.Tensor,
                scheduler: DDIMScheduler, n_steps: int = 50,
                encode_timestep: int = 500) -> torch.Tensor:
    """
    DDIM forward encode: x0 -> x_L in the masked region.

    Parameters
    ----------
    x0              : (1,1,256,256)  input slice on DEVICE
    mask            : (1,1,256,256)  binary mask, 1=pathology region
    scheduler       : DDIMScheduler
    n_steps         : number of DDIM steps for encoding
    encode_timestep : target noise level L (0 to T-1)

    Returns
    -------
    x_L : (1,1,256,256)  noised image at timestep L
    """
    # Direct forward process: add noise to encode_timestep using scheduler
    noise = torch.randn_like(x0)
    t_tensor = torch.tensor([encode_timestep], device=x0.device).long()
    x_noised = scheduler.add_noise(x0, noise, t_tensor)

    # Apply only in masked region; keep original elsewhere
    x_L = x0 * (1 - mask) + x_noised * mask
    return x_L


@torch.no_grad()
def hybrid_decode(x_L: torch.Tensor, x0_orig: torch.Tensor,
                  mask: torch.Tensor,
                  unet: nn.Module,
                  ddpm_sched: DDPMScheduler,
                  ddim_sched: DDIMScheduler,
                  start_t: int = 500,
                  n_ddim_steps: int = 50) -> torch.Tensor:
    """
    Hybrid DDPM/DDIM reverse decode from t=start_t -> t=0.

    Inside  mask (M=1) : DDPM step  — stochastic, allows healthy regeneration
    Outside mask (M=0) : DDIM step  — deterministic, preserves healthy anatomy

    Returns
    -------
    x_hat : (1,1,256,256)  healthy reconstruction
    """
    unet.eval()
    ddim_sched.set_timesteps(n_ddim_steps)

    # Filter timesteps to those <= start_t
    timesteps = [t for t in ddim_sched.timesteps if t <= start_t]

    x = x_L.clone()

    for t_val in timesteps:
        t_batch = torch.tensor([t_val], device=x.device).long()

        noise_pred = unet(x, t_batch)

        # DDPM step (stochastic) for masked region
        ddpm_out = ddpm_sched.step(noise_pred, t_val, x)
        x_ddpm   = ddpm_out.prev_sample

        # DDIM step (deterministic) for non-masked region
        ddim_out = ddim_sched.step(noise_pred, t_val, x)
        x_ddim   = ddim_out.prev_sample

        # Blend: mask region uses DDPM, rest uses DDIM
        x = x_ddim * (1 - mask) + x_ddpm * mask

    return x.clamp(0.0, 1.0)


def anomaly_map(x0: torch.Tensor, x_hat: torch.Tensor,
                smooth_sigma: float = 2.0) -> torch.Tensor:
    """
    Compute pixel-wise anomaly score |x0 - x_hat|.
    Optionally smooth with a Gaussian kernel (sigma in pixels).
    Returns a (1,1,256,256) float32 tensor on CPU.
    """
    diff = (x0 - x_hat).abs().cpu()
    if smooth_sigma > 0:
        # approximate Gaussian with a separable box filter
        k = max(3, int(smooth_sigma * 3) | 1)   # odd kernel size
        pad = k // 2
        kernel = torch.ones(1, 1, 1, k) / k
        diff = F.conv2d(F.conv2d(diff, kernel, padding=(0, pad)),
                        kernel.transpose(-1, -2), padding=(pad, 0))
    return diff

print('Inference functions defined.')


## 10 · Demo inference on a validation slice

Since we have no ACAT classifier yet, we use a **synthetic centre mask**
(a 64×64 square in the middle of the image) to demonstrate the pipeline.
Replace this mask with the ACAT saliency output for real anomaly detection.

In [ ]:
# ── pick one val slice ──────────────────────────────────────────────────
val_batch = next(iter(val_loader))
x0_demo   = val_batch[0:1].to(DEVICE)   # (1, 1, 256, 256)

# ── synthetic mask: 64x64 centre square ─────────────────────────────────
mask_demo = torch.zeros_like(x0_demo)
mask_demo[:, :, 96:160, 96:160] = 1.0   # centre 64x64

ENCODE_T  = 500   # how far to encode (noise level)
N_STEPS   = 50    # DDIM steps for decode

# ── run pipeline ────────────────────────────────────────────────────────
x_L    = ddim_encode(x0_demo, mask_demo, ddim_scheduler,
                     encode_timestep=ENCODE_T)
x_hat  = hybrid_decode(x_L, x0_demo, mask_demo,
                       unet, ddpm_scheduler, ddim_scheduler,
                       start_t=ENCODE_T, n_ddim_steps=N_STEPS)
a_map  = anomaly_map(x0_demo, x_hat, smooth_sigma=2.0)

# ── visualise ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(x0_demo[0, 0].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
axes[0].contour(mask_demo[0, 0].cpu().numpy(), levels=[0.5], colors='red', linewidths=1)
axes[0].set_title('Input x₀\n(red = synthetic mask)', fontsize=11)

axes[1].imshow(x_L[0, 0].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'DDIM encoded x_L\n(t={ENCODE_T}, masked region noised)', fontsize=11)

axes[2].imshow(x_hat[0, 0].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
axes[2].set_title('Reconstruction x̂₀\n(hybrid DDPM/DDIM decode)', fontsize=11)

im = axes[3].imshow(a_map[0, 0].numpy(), cmap='hot', vmin=0)
axes[3].set_title('Anomaly map\n|x₀ - x̂₀|', fontsize=11)
fig.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)

for ax in axes:
    ax.axis('off')

fig.suptitle(
    'NOTE: model is untrained / early-training — reconstruction quality will\n'
    'improve substantially after full training (500+ epochs on GPU)',
    fontsize=10, color='gray'
)
plt.tight_layout()
fig.savefig(LOG_DIR / 'figures' / 'demo_anomaly_pipeline.png', dpi=150)
plt.show()


## 11 · Checkpoint utilities

Two saving strategies:
- **Full checkpoint** — entire model state (needed to resume training)
- **LoRA-only** — only the A/B matrices (~few MB). Load these on top of a
  freshly instantiated base model for inference.

In [ ]:
def save_lora_checkpoint(unet, epoch, path):
    """Save only LoRA weights + training metadata."""
    torch.save({
        'epoch'     : epoch,
        'lora_r'    : LORA_RANK,
        'lora_alpha': LORA_ALPHA,
        'lora_state': lora_state_dict(unet),
    }, path)


def load_lora_checkpoint(unet, path):
    """Load LoRA weights into an existing model (base weights unchanged)."""
    ckpt = torch.load(path, map_location='cpu')
    missing, unexpected = unet.load_state_dict(ckpt['lora_state'], strict=False)
    lora_keys = [k for k in ckpt['lora_state']]
    print(f'Loaded {len(lora_keys)} LoRA tensors from epoch {ckpt["epoch"]}')
    if unexpected:
        print(f'Unexpected keys: {unexpected[:5]}')
    return ckpt['epoch']


# Save final LoRA checkpoint
final_lora_path = CKPT_DIR / 'lora_weights_final.pt'
save_lora_checkpoint(unet, N_EPOCHS, final_lora_path)
print(f'LoRA weights saved to {final_lora_path}')
print(f'File size : {final_lora_path.stat().st_size / 1024:.1f} KB')


## 12 · Next steps

| Step | What to do |
|------|------------|
| **Train longer** | 500–1000 epochs on GPU; monitor val MSE convergence |
| **ACAT classifier** | Train or plug in the saliency/classifier to generate real pathology masks |
| **Tune encode depth** | Try `ENCODE_T` in {250, 500, 750} — higher = more freedom to edit, lower = more preservation |
| **Tune LoRA rank** | `r=4` is conservative; try `r=8` or `r=16` if val loss plateaus |
| **Evaluate anomaly map** | Threshold `a_map` and compute Dice/AUC against BraTS segmentation masks |
| **Multi-step encode** | For smoother masks, run DDIM encode iteratively rather than one-shot |
